# Stage 2 — Logistic Regression + SGD (from scratch)

Train a logistic regression in pure NumPy (sigmoid + BCE + L2, mini-batch SGD) on the Stage 1 features, then evaluate image-level performance and calibration. A **0.45-0.55** band defines a third **suspicious** class.

In [ ]:
from _setup import ARTIFACTS_DIR
import numpy as np
import matplotlib.pyplot as plt
from phytolabs.logreg import LogisticRegressionSGD
from phytolabs.features import FEATURE_NAMES
from phytolabs import viz, metrics, calibration

def load_table(split):
    d = np.load(ARTIFACTS_DIR / f'features_{split}.npz', allow_pickle=True)
    return d['X'], d['y']

X_train, y_train = load_table('train')
X_val, y_val = load_table('val')
print('train', X_train.shape, '| val', X_val.shape)

## Train

In [ ]:
model = LogisticRegressionSGD(lr=0.1, epochs=300, batch_size=16, l2=1e-3, random_state=0)
model.fit(X_train, y_train)
for name, w in zip(FEATURE_NAMES, model.w):
    print(f'{name:22s} {w:+.3f}')
print(f'{"bias":22s} {model.b:+.3f}')

In [ ]:
viz.plot_loss(model.loss_history)
plt.show()

## Evaluate on the validation split

In [ ]:
proba = model.predict_proba(X_val)
pred = model.predict(X_val)
report = metrics.classification_report(y_val, pred, proba)
report

In [ ]:
viz.plot_confusion_matrix(y_val, pred); plt.show()
viz.plot_roc(y_val, proba); plt.show()

## Calibration and the suspicious band

Reliability diagram + how many val images land in each band class.

In [ ]:
viz.plot_reliability(y_val, proba, n_bins=5); plt.show()
print('ECE:', calibration.expected_calibration_error(y_val, proba))
print('band summary:', calibration.band_summary(proba, 0.45, 0.55))

## Persist the trained classifier

In [ ]:
model.save(ARTIFACTS_DIR / 'logreg.joblib')
print('saved', ARTIFACTS_DIR / 'logreg.joblib')